In [1]:
import pandas as pd
import numpy as np

# -----------------------------
# INPUT FILES
# -----------------------------
BASE_PATH = "../data/mdm2_data_files/big_table_with_weather_rain_clusters_cctv300m.csv"
SAFETY_PATH = "../reports/sensor_ward_safety_after_dark_2024.csv"

OUT_PATH = "../data/mdm2_data_files/master_mobility_model_data.csv"

# -----------------------------
# LOAD
# -----------------------------
df = pd.read_csv(BASE_PATH)
safety = pd.read_csv(SAFETY_PATH)

print("Base shape:", df.shape)
print("Safety shape:", safety.shape)

# -----------------------------
# BASIC DATETIME FEATURES
# -----------------------------
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

df["week"] = df["datetime"].dt.isocalendar().week.astype("Int64")
df["year"] = df["datetime"].dt.year
df["is_weekend"] = df["dow"].isin([5, 6]).astype(int)

# -----------------------------
# ACTIVE TRAVEL
# -----------------------------
df["active"] = df["ped"] + df["cyc"]

# -----------------------------
# CLEAN SAFETY FILE
# -----------------------------
safety = safety.copy()
safety["sensor_id"] = safety["sensor_id"].astype(int)

# detect safety column
if "safety_after_dark_pct" not in safety.columns:
    raise ValueError("Expected safety_after_dark_pct in safety file.")

# scale safety 0-1
smin = safety["safety_after_dark_pct"].min()
smax = safety["safety_after_dark_pct"].max()

safety["safety_scaled"] = (safety["safety_after_dark_pct"] - smin) / (smax - smin)

safety_small = safety[["sensor_id", "ward_name", "safety_after_dark_pct", "safety_scaled"]].drop_duplicates()

# -----------------------------
# MERGE SAFETY
# -----------------------------
df["sensor_id"] = df["sensor_id"].astype(int)
df = df.merge(safety_small, on="sensor_id", how="left")

print("After safety merge:", df.shape)
print("Missing safety_scaled rate:", df["safety_scaled"].isna().mean())

# -----------------------------
# ORDER COLUMNS NICELY
# -----------------------------
preferred_order = [
    "sensor_id", "datetime", "date_only", "hour", "dow", "is_weekend", "week", "month", "year",
    "longitude", "latitude", "cluster_geo", "ward_name",
    "ped", "cyc", "car", "active",
    "solar_altitude_deg", "light_class", "Dark",
    "temp_c", "wind_ms", "rain_mm",
    "cctv_300m", "safety_after_dark_pct", "safety_scaled"
]

cols_present = [c for c in preferred_order if c in df.columns]
cols_rest = [c for c in df.columns if c not in cols_present]
df = df[cols_present + cols_rest]

# -----------------------------
# SAVE
# -----------------------------
df.to_csv(OUT_PATH, index=False)

print("Saved master dataset to:")
print(OUT_PATH)
print("\nColumns:")
print(df.columns.tolist())
print("\nUnique sensors:", df["sensor_id"].nunique())
print("Rows:", len(df))

/Users/shavarshmelikyan/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Base shape: (379477, 20)
Safety shape: (46, 3)
After safety merge: (379477, 27)
Missing safety_scaled rate: 0.0
Saved master dataset to:
../data/mdm2_data_files/master_mobility_model_data.csv

Columns:
['sensor_id', 'datetime', 'date_only', 'hour', 'dow', 'is_weekend', 'week', 'month', 'year', 'longitude', 'latitude', 'cluster_geo', 'ward_name', 'ped', 'cyc', 'car', 'active', 'solar_altitude_deg', 'light_class', 'Dark', 'temp_c', 'wind_ms', 'rain_mm', 'cctv_300m', 'safety_after_dark_pct', 'safety_scaled', 'weekday']

Unique sensors: 46
Rows: 379477
